In [57]:
import os
import json
import time
from pathlib import Path

import requests
import pandas as pd
import time

тут короче подключаюсь к API product hunt, беру их токен через оф сайт для разработки (у них классный, можно зарегатся и все сразу предоставляют бесплатно) и за тем настраиваю

In [50]:
BASE_URL = "https://api.producthunt.com/v2/api/graphql"
TOKEN_FROM_ENV = os.getenv("PRODUCT_HUNT_TOKEN")

ACCESS_TOKEN = "MvNc1VujewxyZL1pMIh9B6eDTq89hgzjT6cMbRV3sU8"

if not ACCESS_TOKEN or ACCESS_TOKEN == "PASTE_YOUR_PRODUCT_HUNT_TOKEN_HERE":
    raise ValueError(
        "Не найден PRODUCT_HUNT_TOKEN. "
        "Либо задай os.environ['PRODUCT_HUNT_TOKEN'], либо вставь токен в ACCESS_TOKEN."
    )

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

OUT_DIR = Path("product_hunt_large_output")
OUT_DIR.mkdir(exist_ok=True)

TARGET_ROWS = 1000

POSTS_PAGE_SIZE = 20
TOPICS_PAGE_SIZE = 50
COLLECTIONS_PAGE_SIZE = 50
COMMENT_IDS_POSTS_PAGE_SIZE = 5
COMMENT_IDS_PER_POST = 5

MAX_POST_PAGES = 200
MAX_TOPIC_PAGES = 100
MAX_COLLECTION_PAGES = 100
MAX_COMMENT_ID_PAGES = 400

SLEEP_SECONDS = 0.35
USER_SLEEP_SECONDS = 0.20
COMMENT_SLEEP_SECONDS = 0.20

REQUEST_TIMEOUT = 40
RETRIES = 3


и добавил ограничение по запросам, но часть дальше менял много раз, потому что словил большое количество блокировки от слишком частой частоты запросов, часть ограничел по объему так как очень долго выполнялось и главным условием сделал 5 разных типов запроса

In [3]:

def run_query(query: str, variables: dict | None = None, retries: int = RETRIES):
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                BASE_URL,
                headers=HEADERS,
                json={"query": query, "variables": variables or {}},
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            payload = response.json()
            if "errors" in payload:
                raise RuntimeError(json.dumps(payload["errors"], ensure_ascii=False, indent=2))
            return payload["data"]
        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.0 * attempt)
            else:
                raise last_error


1) Для Posts

Взял посты, чтобы первично на сайт проанализировать посты, тут было не слишком много вариантов, что анализировать, так как у данного апи ограничены данные, которые можно запрашивать

In [4]:

QUERY_POSTS = '''
query($first: Int!, $after: String) {
  posts(first: $first, after: $after) {
    pageInfo {
      hasNextPage
      endCursor
    }
    edges {
      node {
        id
        name
        slug
        tagline
        votesCount
        commentsCount
        reviewsCount
        reviewsRating
        createdAt
        featuredAt
        url
        website
        user {
          username
        }
      }
    }
  }
}
'''

def flatten_post(node: dict) -> dict:
    user = node.get("user") or {}
    return {
        "post_id": node.get("id"),
        "post_name": node.get("name"),
        "slug": node.get("slug"),
        "tagline": node.get("tagline"),
        "votes_count": node.get("votesCount"),
        "comments_count": node.get("commentsCount"),
        "reviews_count": node.get("reviewsCount"),
        "reviews_rating": node.get("reviewsRating"),
        "created_at": node.get("createdAt"),
        "featured_at": node.get("featuredAt"),
        "post_url": node.get("url"),
        "website": node.get("website"),
        "author_username": user.get("username"),
    }

posts_rows = []
posts_raw = []
after_cursor = None

for page in range(1, MAX_POST_PAGES + 1):
    data = run_query(QUERY_POSTS, {"first": POSTS_PAGE_SIZE, "after": after_cursor})
    block = data["posts"]
    posts_raw.append(data)

    for edge in block["edges"]:
        posts_rows.append(flatten_post(edge["node"]))

    print(f"posts: page={page}, total_raw_rows={len(posts_rows)}")

    if len(posts_rows) >= TARGET_ROWS:
        break
    if not block["pageInfo"]["hasNextPage"]:
        break

    after_cursor = block["pageInfo"]["endCursor"]
    time.sleep(SLEEP_SECONDS)

df_posts = pd.DataFrame(posts_rows).drop_duplicates(subset=["post_id"]).reset_index(drop=True)
print("posts unique rows:", len(df_posts))
df_posts.head()


posts: page=1, total_raw_rows=20
posts: page=2, total_raw_rows=40
posts: page=3, total_raw_rows=60
posts: page=4, total_raw_rows=80
posts: page=5, total_raw_rows=100
posts: page=6, total_raw_rows=120
posts: page=7, total_raw_rows=140
posts: page=8, total_raw_rows=160
posts: page=9, total_raw_rows=180
posts: page=10, total_raw_rows=200
posts: page=11, total_raw_rows=220
posts: page=12, total_raw_rows=240
posts: page=13, total_raw_rows=260
posts: page=14, total_raw_rows=280
posts: page=15, total_raw_rows=300
posts: page=16, total_raw_rows=320
posts: page=17, total_raw_rows=340
posts: page=18, total_raw_rows=360
posts: page=19, total_raw_rows=380
posts: page=20, total_raw_rows=400
posts: page=21, total_raw_rows=420
posts: page=22, total_raw_rows=440
posts: page=23, total_raw_rows=460
posts: page=24, total_raw_rows=480
posts: page=25, total_raw_rows=500
posts: page=26, total_raw_rows=520
posts: page=27, total_raw_rows=540
posts: page=28, total_raw_rows=560
posts: page=29, total_raw_rows=58

,post_id,post_name,slug,tagline,votes_count,comments_count,reviews_count,reviews_rating,created_at,featured_at,post_url,website,author_username
0,1120099,Claude Advisor tool,claude-advisor-tool,Pair Opus as advisor with Sonnet or Haiku as e...,242,8,720,4.97,2026-04-10T07:01:00Z,2026-04-10T07:01:00Z,https://www.producthunt.com/products/claude?ut...,https://www.producthunt.com/r/GJ45R3YKKOWWA5?u...,rohanrecommends
1,1118052,Integrations in Spine,integrations-in-spine,AI that synthesize and researches info across ...,235,33,2,5.00,2026-04-10T07:01:00Z,2026-04-10T07:01:00Z,https://www.producthunt.com/products/spine-2?u...,https://www.producthunt.com/r/VYT2POZCTPRQOT?u...,rohanrecommends
2,1119001,Google Finance,google-finance-3,"Ask complex finance questions, get AI-grounded...",190,3,69,4.86,2026-04-10T07:01:00Z,2026-04-10T07:01:00Z,https://www.producthunt.com/products/google?ut...,https://www.producthunt.com/r/GNTCPZ4KQWJ6AP?u...,rohanrecommends
3,1120270,Drift,drift-a7eecd58-2a1d-4550-8750-d37c3edb2791,Minimalist journal app that fades over time,167,34,0,0.00,2026-04-10T07:01:00Z,2026-04-10T07:01:00Z,https://www.producthunt.com/products/drift-jou...,https://www.producthunt.com/r/77I2D7OKQILF7O?u...,letitdrift
4,1119130,SoulLink,soullink-e80a20ab-f001-437e-8185-f9ec12e49a27,Mobile 3D Vibe Coding Partner,155,9,0,0.00,2026-04-10T07:01:00Z,2026-04-10T07:01:00Z,https://www.producthunt.com/products/soullinka...,https://www.producthunt.com/r/WDAZDHMMZORTEO?u...,michael_shang


In [13]:

df_posts.to_csv(OUT_DIR / "posts1.csv", index=False)


with open(OUT_DIR / "posts_raw1.json", "w", encoding="utf-8") as f:
    json.dump(posts_raw, f, ensure_ascii=False, indent=2)





2) Для Topics

тут решил посмотреть темы, чтобы понять, в каких категориях лежат продукты, пошел немного другим путем по сбору данных

In [10]:
QUERY_TOPICS = '''
query($first: Int!, $after: String) {
  topics(first: $first, after: $after) {
    pageInfo {
      hasNextPage
      endCursor
    }
    edges {
      node {
        id
        name
        slug
        followersCount
      }
    }
  }
}
'''

topics_rows = []
topics_raw = []
after_cursor = None

for page in range(1, MAX_TOPIC_PAGES + 1):
    data = run_query(QUERY_TOPICS, {"first": TOPICS_PAGE_SIZE, "after": after_cursor})
    block = data["topics"]
    topics_raw.append(data)

    for edge in block["edges"]:
        node = edge["node"]
        topics_rows.append({
            "topic_id": node.get("id"),
            "topic_name": node.get("name"),
            "topic_slug": node.get("slug"),
            "followers_count": node.get("followersCount"),
        })

    current_unique = len(pd.DataFrame(topics_rows).drop_duplicates(subset=["topic_id"]))
    print(f"topics: page={page}, current_unique={current_unique}")

    if current_unique >= TARGET_ROWS:
        break
    if not block["pageInfo"]["hasNextPage"]:
        break

    after_cursor = block["pageInfo"]["endCursor"]
    time.sleep(3)

df_topics = pd.DataFrame(topics_rows).drop_duplicates(subset=["topic_id"]).reset_index(drop=True)
print("topics unique rows:", len(df_topics))
df_topics.head()


topics: page=1, current_unique=20
topics: page=2, current_unique=40
topics: page=3, current_unique=60
topics: page=4, current_unique=80
topics: page=5, current_unique=100
topics: page=6, current_unique=120
topics: page=7, current_unique=140
topics: page=8, current_unique=160
topics: page=9, current_unique=180
topics: page=10, current_unique=200
topics: page=11, current_unique=220
topics: page=12, current_unique=240
topics: page=13, current_unique=260
topics: page=14, current_unique=280
topics: page=15, current_unique=300
topics: page=16, current_unique=320
topics: page=17, current_unique=340
topics: page=18, current_unique=360
topics: page=19, current_unique=380
topics: page=20, current_unique=400
topics: page=21, current_unique=420
topics: page=22, current_unique=425
topics unique rows: 425


,topic_id,topic_name,topic_slug,followers_count
0,1293,Pitch Paris,pitch-paris,0
1,1260,Alpha,alpha,10
2,1227,YC Application,yc-application,33
3,1194,Vibe coding,vibe-coding,379
4,1167,Apple Vision Pro,apple-vision-pro,192


In [14]:

df_topics.to_csv(OUT_DIR / "topics1.csv", index=False)



with open(OUT_DIR / "topics_raw1.json", "w", encoding="utf-8") as f:
    json.dump(topics_raw, f, ensure_ascii=False, indent=2)



3) Для Collections

Решил проанализировать подборки продуктов, чтобы углубится в дальнешею бизнес-часть, тут собственно говоря собрал по ней данные

In [18]:

QUERY_COLLECTIONS = '''
query($first: Int!, $after: String) {
  collections(first: $first, after: $after) {
    pageInfo {
      hasNextPage
      endCursor
    }
    edges {
      node {
        id
        name
        description
        followersCount
        url
      }
    }
  }
}
'''

collections_rows = []
collections_raw = []
after_cursor = None

for page in range(1, MAX_COLLECTION_PAGES + 1):
    data = run_query(QUERY_COLLECTIONS, {"first": COLLECTIONS_PAGE_SIZE, "after": after_cursor})
    block = data["collections"]
    collections_raw.append(data)

    for edge in block["edges"]:
        node = edge["node"]
        collections_rows.append({
            "collection_id": node.get("id"),
            "collection_name": node.get("name"),
            "tagline": node.get("tagline"),
            "description": node.get("description"),
            "followers_count": node.get("followersCount"),
            "collection_url": node.get("url"),
        })

    current_unique = len(pd.DataFrame(collections_rows).drop_duplicates(subset=["collection_id"]))
    print(f"collections: page={page}, current_unique={current_unique}")

    if current_unique >= TARGET_ROWS:
        break
    if not block["pageInfo"]["hasNextPage"]:
        break

    after_cursor = block["pageInfo"]["endCursor"]
    time.sleep(SLEEP_SECONDS)

df_collections = pd.DataFrame(collections_rows).drop_duplicates(subset=["collection_id"]).reset_index(drop=True)
print("collections unique rows:", len(df_collections))
df_collections.head()


collections: page=1, current_unique=20
collections: page=2, current_unique=40
collections: page=3, current_unique=60
collections: page=4, current_unique=80
collections: page=5, current_unique=100
collections: page=6, current_unique=120
collections: page=7, current_unique=140
collections: page=8, current_unique=160
collections: page=9, current_unique=180
collections: page=10, current_unique=200
collections: page=11, current_unique=220
collections: page=12, current_unique=240
collections: page=13, current_unique=260
collections: page=14, current_unique=280
collections: page=15, current_unique=300
collections: page=16, current_unique=320
collections: page=17, current_unique=340
collections: page=18, current_unique=360
collections: page=19, current_unique=380
collections: page=20, current_unique=399
collections: page=21, current_unique=419
collections: page=22, current_unique=439
collections: page=23, current_unique=459
collections: page=24, current_unique=479
collections: page=25, current

,collection_id,collection_name,tagline,description,followers_count,collection_url
0,8890,Free Stuff For Startups,None,"Free stuff, tools and products with free plans...",4829,https://www.producthunt.com/@hnshah/collection...
1,160644,Brain Music,None,A collection of music products that help you f...,4449,https://www.producthunt.com/collections/brain-...
2,183205,$0 Design Tools,None,A collection of $0 Design Tools to Help You Cr...,4293,https://www.producthunt.com/collections/0-desi...
3,239660,Meditation Tools ☺️,None,"Apps, products, and hardware to help you relax...",3735,https://www.producthunt.com/collections/medita...
4,50210,Remote Apps,None,The best apps for remote work wherever you are...,3698,https://www.producthunt.com/collections/remote...


In [32]:

df_collections.to_csv(OUT_DIR / "collections1.csv", index=False)

with open(OUT_DIR / "collections_raw1.json", "w", encoding="utf-8") as f:
    json.dump(collections_raw, f, ensure_ascii=False, indent=2)


4) Для Users

с пользователями оказалось не все так просто, user_id приходит странный (часто 0)

In [58]:
QUERY_USER = '''
query($username: String!) {
  user(username: $username) {
    id
    name
    username
    headline
    followersCount
    followingCount
    createdAt
    isMaker
    twitterUsername
    websiteUrl
    url
  }
}
'''

TARGET_USER_ROWS = 40
MAX_USER_CHECKS = 300
USER_SLEEP_SECONDS = 0.5

candidate_usernames = df_posts["author_username"].dropna().tolist()
candidate_usernames = [str(u).strip() for u in candidate_usernames if str(u).strip()]
candidate_usernames = list(dict.fromkeys(candidate_usernames))

print(f"users: total_candidates={len(candidate_usernames)}")

user_rows = []
users_raw = []
failed_usernames = []
checked_count = 0

for username in candidate_usernames:
    if len(user_rows) >= TARGET_USER_ROWS:
        break
    if checked_count >= MAX_USER_CHECKS:
        break

    checked_count += 1

    try:
        data = run_query(QUERY_USER, {"username": username})
        users_raw.append({
            "requested_username": username,
            "response": data
        })

        node = data.get("user")

        if not node:
            failed_usernames.append(username)
            continue

        user_rows.append({
            "requested_username": username,
            "user_id_raw": node.get("id"),
            "name": node.get("name"),
            "username_returned": node.get("username"),
            "headline": node.get("headline"),
            "followers_count": node.get("followersCount"),
            "following_count": node.get("followingCount"),
            "created_at": node.get("createdAt"),
            "is_maker": node.get("isMaker"),
            "twitter_username": node.get("twitterUsername"),
            "website_url": node.get("websiteUrl"),
            "profile_url": node.get("url"),
        })

    except Exception:
        failed_usernames.append(username)

    if checked_count % 25 == 0:
        print(
            f"users: checked={checked_count}, current_rows={len(user_rows)}, failed={len(failed_usernames)}"
        )
        pd.DataFrame(user_rows).drop_duplicates(subset=["requested_username"]).to_csv(
            OUT_DIR / "user_partial.csv", index=False
        )

    time.sleep(USER_SLEEP_SECONDS)

df_user = (
    pd.DataFrame(user_rows)
    .drop_duplicates(subset=["requested_username"])
    .reset_index(drop=True)
)

print("users rows:", len(df_user))
print("checked usernames:", checked_count)
print("failed usernames:", len(failed_usernames))

df_user.head()

users: total_candidates=741
users: checked=25, current_rows=23, failed=2
users: checked=50, current_rows=23, failed=27
users: checked=75, current_rows=23, failed=52
users: checked=100, current_rows=23, failed=77
users: checked=125, current_rows=23, failed=102
users: checked=150, current_rows=31, failed=119
users rows: 40
checked usernames: 159
failed usernames: 119


,requested_username,user_id_raw,name,username_returned,headline,followers_count,following_count,created_at,is_maker,twitter_username,website_url,profile_url
0,rohanrecommends,0,[REDACTED],[REDACTED],None,13728,4138,None,False,None,None,[REDACTED]
1,letitdrift,0,[REDACTED],[REDACTED],None,31,25,None,False,None,None,[REDACTED]
2,michael_shang,0,[REDACTED],[REDACTED],None,62,19,None,False,None,None,[REDACTED]
3,p5ina,0,[REDACTED],[REDACTED],None,26,4,None,False,None,None,[REDACTED]
4,fmerian,0,[REDACTED],[REDACTED],None,58038,28,None,False,None,None,[REDACTED]


In [59]:
df_user.to_csv(OUT_DIR / "user1.csv", index=False)

with open(OUT_DIR / "user1_raw.json", "w", encoding="utf-8") as f:
    json.dump(users_raw, f, ensure_ascii=False, indent=2)


5) Для Comments

отдельно собираю id комментариев, потому что API позволяет брать комментарий только по id, и уже по ним дополнительно смотрю сами коменты

In [44]:
QUERY_POSTS_COMMENT_IDS = '''
query($first: Int!, $after: String) {
  posts(first: $first, after: $after) {
    pageInfo {
      hasNextPage
      endCursor
    }
    edges {
      node {
        id
        comments(first: 5) {
          edges {
            node {
              id
            }
          }
        }
      }
    }
  }
}
'''

COMMENT_ID_PAGE_SIZE = 10
MAX_COMMENT_ID_PAGES = 10
COMMENT_ID_SLEEP_SECONDS = 0.7

comment_ids = []
after_cursor = None

for page in range(1, MAX_COMMENT_ID_PAGES + 1):
    data = run_query(
        QUERY_POSTS_COMMENT_IDS,
        {"first": COMMENT_ID_PAGE_SIZE, "after": after_cursor}
    )

    block = data["posts"]

    for edge in block["edges"]:
        node = edge["node"]
        comments_block = node.get("comments") or {}
        comment_edges = comments_block.get("edges") or []

        for cedge in comment_edges:
            cnode = cedge.get("node") or {}
            cid = cnode.get("id")
            if cid:
                comment_ids.append(cid)

    comment_ids = list(dict.fromkeys(comment_ids))
    print(f"comment_ids: page={page}, current_unique={len(comment_ids)}")

    if not block["pageInfo"]["hasNextPage"]:
        break

    after_cursor = block["pageInfo"]["endCursor"]
    time.sleep(COMMENT_ID_SLEEP_SECONDS)

print("total unique comment ids:", len(comment_ids))
print(comment_ids[:20])

comment_ids: page=1, current_unique=39
comment_ids: page=2, current_unique=70
comment_ids: page=3, current_unique=96
comment_ids: page=4, current_unique=113
comment_ids: page=5, current_unique=134
comment_ids: page=6, current_unique=148
comment_ids: page=7, current_unique=158
comment_ids: page=8, current_unique=166
comment_ids: page=9, current_unique=171
comment_ids: page=10, current_unique=177
total unique comment ids: 177
['5285983', '5285558', '5285498', '5285226', '5285194', '5283254', '5285645', '5285626', '5285574', '5285557', '5284188', '5284117', '5284096', '5283841', '5285293', '5285277', '5285202', '5285189', '5278404', '5286050']


часть нормально не достается и только беру рабочие коменты

In [55]:
QUERY_COMMENT = '''
query($id: ID!) {
  comment(id: $id) {
    id
    body
    createdAt
    userId

    votesCount
    url
    user {
      id
      name
      username
    }
  }
}
'''

TARGET_COMMENT_ROWS = 40
MAX_COMMENT_CHECKS = 200
COMMENT_SLEEP_SECONDS = 0.5

comment_ids = [str(cid).strip() for cid in comment_ids if str(cid).strip()]
comment_ids = list(dict.fromkeys(comment_ids))

print(f"comments: total_candidates={len(comment_ids)}")

comment_rows = []
comments_raw = []
failed_comment_ids = []
checked_count = 0

for cid in comment_ids:
    if len(comment_rows) >= TARGET_COMMENT_ROWS:
        break
    if checked_count >= MAX_COMMENT_CHECKS:
        break

    checked_count += 1

    try:
        data = run_query(QUERY_COMMENT, {"id": cid})
        comments_raw.append(data)

        node = data.get("comment")

        if not node:
            failed_comment_ids.append(cid)
            continue

        user = node.get("user") or {}

        comment_rows.append({
            "requested_comment_id": cid,
            "comment_id_raw": node.get("id"),
            "body": node.get("body"),
            "created_at": node.get("createdAt"),
            "user_id_raw": node.get("userId"),
            "parent_id": node.get("parentId"),
            "votes_count": node.get("votesCount"),
            "comment_url": node.get("url"),
            "author_profile_id_raw": user.get("id"),
            "author_name": user.get("name"),
            "author_username": user.get("username"),
        })

    except Exception:
        failed_comment_ids.append(cid)

    if checked_count % 25 == 0:
        print(
            f"comments: checked={checked_count}, current_rows={len(comment_rows)}, failed={len(failed_comment_ids)}"
        )
        pd.DataFrame(comment_rows).drop_duplicates(subset=["requested_comment_id"]).to_csv("comment_partial.csv", index=False)

    time.sleep(COMMENT_SLEEP_SECONDS)

df_comment = (
    pd.DataFrame(comment_rows)
    .drop_duplicates(subset=["requested_comment_id"])
    .reset_index(drop=True)
)

print("comments rows:", len(df_comment))
print("checked comment ids:", checked_count)
print("failed comment ids:", len(failed_comment_ids))

df_comment.head()

comments: total_candidates=177
comments: checked=25, current_rows=25, failed=0
comments rows: 40
checked comment ids: 40
failed comment ids: 0


,requested_comment_id,comment_id_raw,body,created_at,user_id_raw,parent_id,votes_count,comment_url,author_profile_id_raw,author_name,author_username
0,5285983,5285983,<p>This was the day of the year that PH got pu...,2026-04-10T21:46:03Z,44118,None,0,https://www.producthunt.com/products/claude?co...,0,[REDACTED],[REDACTED]
1,5285558,5285558,<pre><code>Interesting setup — separating advi...,2026-04-10T17:21:53Z,9653791,None,0,https://www.producthunt.com/products/claude?co...,0,[REDACTED],[REDACTED]
2,5285498,5285498,<p>I have a question: Are these Claude tools h...,2026-04-10T16:38:30Z,9668421,None,0,https://www.producthunt.com/products/claude?co...,0,[REDACTED],[REDACTED]
3,5285226,5285226,<p>This is a smart pattern for production agen...,2026-04-10T14:33:32Z,9647446,None,0,https://www.producthunt.com/products/claude?co...,0,[REDACTED],[REDACTED]
4,5285194,5285194,<p>From my experience running multi-agent work...,2026-04-10T14:19:35Z,9645279,None,0,https://www.producthunt.com/products/claude?co...,0,[REDACTED],[REDACTED]


In [56]:
df_comment.to_csv(OUT_DIR / "comment1.csv", index=False)
with open(OUT_DIR / "comment1_raw.json", "w", encoding="utf-8") as f:
    json.dump(comments_raw, f, ensure_ascii=False, indent=2)